In [1]:
import os
import torch
from pymilvus import connections, Collection
from transformers import AutoModel, AutoTokenizer
import ollama

# ========================
# CONFIGURAÇÃO DO MILVUS
# ========================
connections.connect("default", host="127.0.0.1", port="19530")
COLLECTION_NAME = "rag_embeddings_milvus"
collection = Collection(COLLECTION_NAME)

# ========================
# EMBEDDINGS (mesmo modelo usado na ingestão)
# ========================
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].numpy().tolist()

# ========================
# FUNÇÃO DE RECUPERAÇÃO DO CONTEXTO
# ========================
def retrieve_context(query: str, top_k: int = 5):
    query_emb = get_embedding(query)

    collection.load()
    results = collection.search(
        data=[query_emb],
        anns_field="embedding",
        param={"metric_type": "IP", "params": {"nprobe": 10}},
        limit=top_k,
        output_fields=["source_file", "source_url", "chunk_index", "chunk_text"]
    )

    contexts = []
    refs = []
    for r in results[0]:
        chunk_text = r.entity.get("chunk_text")
        source = r.entity.get("source_file")
        url = r.entity.get("source_url")
        contexts.append(chunk_text)
        refs.append(f"📄 {source} | 🔗 {url}")

    return "\n\n".join(contexts), "\n".join(refs)
    
def retrieve_images(query: str, top_k: int = 5):
    query_emb = get_embedding(query)

    image_collection = Collection("image_descriptions")
    image_collection.load()

    results = image_collection.search(
        data=[query_emb],
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": 10}},  # <- corrigido aqui
        limit=top_k,
        output_fields=["url", "titles", "texts", "category"]
    )

    images = []
    for r in results[0]:
        url = r.entity.get("url")
        titles = r.entity.get("titles")
        texts = r.entity.get("texts")
        category = r.entity.get("category")
        score = r.distance
        images.append(
            f"🖼️ {category} | {titles} | {texts[:80]}... ({url}) [score={score:.3f}]"
        )

    return "\n".join(images)



# ========================
# FUNÇÃO DE GERAÇÃO DE RESPOSTA (via Ollama + Mistral)
# ========================
def generate_answer(query: str, context: str):
    prompt = f"""
Você é um assistente técnico especializado em licenciamento ambiental (EIA/RIMA).
Responda à pergunta do usuário **usando apenas o contexto fornecido**.

Contexto:
{context}

Pergunta:
{query}

Responda de forma clara, objetiva e técnica.
"""
    response = ollama.chat(
        model="mistral:7b",
        messages=[
            {"role": "system", "content": "Você é um assistente técnico ambiental especializado em EIA/RIMA."},
            {"role": "user", "content": prompt}
        ]
    )
    return response["message"]["content"]

# ========================
# LOOP DE CHAT
# ========================
print("🤖 Chatbot EIA/RIMA usando Milvus + Mistral 7B (Ollama). Digite 'sair' para encerrar.\n")

while True:
    user_input = input("Você: ")
    if user_input.lower() in ["sair", "exit", "quit"]:
        break

    # Recupera contexto textual
    context, refs = retrieve_context(user_input)

    # Recupera imagens relacionadas
    images = retrieve_images(user_input)

    # Gera resposta
    answer = generate_answer(user_input, context)

    print("\nBot:", answer)
    print("\n--- Fontes ---")
    print(refs)

    if images:
        print("\n--- Imagens relacionadas ---")
        print(images)
    print("contexto: ")
    print (context)


🤖 Chatbot EIA/RIMA usando Milvus + Mistral 7B (Ollama). Digite 'sair' para encerrar.



Você:  como uma pedreira afeta o meio ambiente?



Bot:  Uma pedreira pode afectar o meio ambiente por vários impactos que podem ser identificados nas fases de planejamento, implantação e operação. Alguns dos principais impactos incluem:
1. Interferências na fauna silvestre, causando perturbações, deslocamentos e afugentamentos, alterações nos hábitos e consequentemente expondo os animais a riscos de acidentes e confrontos com funcionários, submetendo-os às condições de estresse.
2. Atração dos animais para perto das câmeras, causada pela introdução de alimentos não naturais (como frutas ou proteínas animais) durante a fase de planejamento (levantamento), que impactam a fauna terrestre.
3. Captura e manejo dos peixes para o levantamento, causando impactos sobre a ictiofauna na fase de planejamento.
4. Alterações na estrutura física e química dos ambientes aquáticos e suas áreas adjacências, afetando integridade das comunidades de peixes.
5. Impactos decorrentes da geração excessiva de ruídos e atropelamentos dos animais presentes na á

Você:  sair
